In [4]:
from dash import Dash, dash_table, html, dcc
from dash import Input, Output, ctx
import pandas as pd
import json

app = Dash(__name__)

# ------------------------------------------------------------------
# Sample Data
# Each record has a unique 'id' for easy reference.
# ------------------------------------------------------------------
df = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106, 107, 108],
    "Name": ["Ali", "Sara", "Reza", "Maryam", "Tom", "Zhina", "Boshra", "Jack"],
    "Age": [20, 31, 28, 24, 26, 33, 21, 29],
    "Score": [80, 95, 76, 88, 92, 70, 85, 99]
})

app.layout = html.Div([
    dash_table.DataTable(
        id="tbl",
        data=df.to_dict("records"),
        columns=[
            {"name": "ID", "id": "id"},
            {"name": "Name", "id": "Name"},
            {"name": "Age", "id": "Age"},
            {"name": "Score", "id": "Score"},
        ],
        editable=True,
        row_selectable="multi",
        column_selectable="multi",
        sort_action="native",
        filter_action="native",
        page_action="native",
        page_size=4,
        style_table={'overflowX': 'auto'},
    ),

    html.Hr(),
    html.H3("DataTable Inspector", style={'textAlign': 'center'}),

    # Container for the inspector output
    html.Div(id="debug", style={
        'maxHeight': '70vh',
        'overflow': 'auto',
        'padding': '15px',
        'backgroundColor': '#f8f9fa',
        'border': '1px solid #ddd',
        'borderRadius': '8px'
    })
])


def create_section(title, content):
    """Create a collapsible section for better readability."""
    return html.Details([
        html.Summary(title, style={
            'fontWeight': 'bold', 
            'fontSize': '16px', 
            'cursor': 'pointer'
        }),
        html.Div(content, style={
            'padding': '10px 20px', 
            'backgroundColor': 'white', 
            'marginBottom': '10px',
            'borderRadius': '6px'
        })
    ], style={'marginBottom': '8px', 'border': '1px solid #eee', 'borderRadius': '6px'})


@app.callback(
    Output("debug", "children"),
    [Input("tbl", prop) for prop in [
        "data", "data_previous", "columns",
        "selected_rows", "selected_row_ids",
        "selected_columns", "active_cell", "selected_cells",
        "start_cell", "end_cell",
        "derived_virtual_data", "derived_virtual_indices", "derived_virtual_row_ids",
        "derived_virtual_selected_rows", "derived_virtual_selected_row_ids",
        "derived_viewport_data", "derived_viewport_indices", "derived_viewport_row_ids",
        "derived_viewport_selected_rows", "derived_viewport_selected_row_ids",
    ]]
)
def inspector(
    data, data_previous, columns,
    selected_rows, selected_row_ids,
    selected_columns, active_cell, selected_cells,
    start_cell, end_cell,
    derived_virtual_data, derived_virtual_indices, derived_virtual_row_ids,
    derived_virtual_selected_rows, derived_virtual_selected_row_ids,
    derived_viewport_data, derived_viewport_indices, derived_viewport_row_ids,
    derived_viewport_selected_rows, derived_viewport_selected_row_ids,
):
    """
    Main callback that displays all important DataTable properties
    in a user-friendly format instead of raw JSON.
    """

    # Determine what triggered the callback
    trigger = ctx.triggered_id
    trigger_property = ctx.triggered[0]["prop_id"] if ctx.triggered else None

    # Get clicked row data if available
    clicked_row_data = None
    if active_cell and active_cell.get("row_id") is not None:
        for row in data or []:
            if row.get("id") == active_cell["row_id"]:
                clicked_row_data = row
                break

    sections = []

    # 1. Trigger Information
    sections.append(create_section("🔄 Trigger Information", [
        html.P([html.Strong("Component: "), str(trigger)]),
        html.P([html.Strong("Property: "), str(trigger_property)]),
    ]))

    # 2. Clicked / Active Row
    sections.append(create_section("👆 Clicked / Active Row", [
        html.P([html.Strong("Row Index: "), str(active_cell["row"] if active_cell else None)]),
        html.P([html.Strong("Row ID: "), str(active_cell.get("row_id") if active_cell else None)]),
        html.H6("Row Data:"),
        dash_table.DataTable(
            data=[clicked_row_data] if clicked_row_data else [],
            columns=[{"name": k, "id": k} for k in (clicked_row_data.keys() if clicked_row_data else [])],
            style_cell={'textAlign': 'left'},
        ) if clicked_row_data else html.P("No row clicked", style={'color': 'gray'})
    ]))

    # 3. Active Cell & Selected Cells
    sections.append(create_section("📍 Active Cell & Selected Cells", [
        html.Pre(json.dumps(active_cell, indent=2, ensure_ascii=False), style={'backgroundColor': '#f0f0f0'}) 
            if active_cell else html.P("None"),
        html.H6("Selected Cells:"),
        html.Pre(json.dumps(selected_cells, indent=2, ensure_ascii=False) if selected_cells else "[]"),
    ]))

    # 4. Selected Rows
    sections.append(create_section("📋 Selected Rows", [
        html.P([html.Strong("Selected by Index: "), str(selected_rows)]),
        html.P([html.Strong("Selected by ID: "), str(selected_row_ids)]),
    ]))

    # 5. Selected Columns
    sections.append(create_section("🏛️ Selected Columns", [
        html.P(str(selected_columns) if selected_columns else "None")
    ]))

    # 6. Table Columns Definition
    sections.append(create_section("📊 Table Columns", [
        dash_table.DataTable(
            data=columns,
            columns=[{"name": "Name", "id": "name"}, {"name": "ID", "id": "id"}],
            style_cell={'textAlign': 'left'}
        )
    ]))

    # 7. Current Viewport (Visible Page)
    sections.append(create_section("👁️ Current Viewport Data (Current Page)", [
        dash_table.DataTable(
            data=derived_viewport_data or [],
            columns=[{"name": c["name"], "id": c["id"]} for c in columns] if columns else [],
            page_size=10,
            style_table={'overflowX': 'auto'}
        )
    ]))

    # 8. Virtual Data (After filter/sort)
    sections.append(create_section("🔎 Virtual Data (After Filter / Sort)", [
        dash_table.DataTable(
            data=derived_virtual_data or [],
            columns=[{"name": c["name"], "id": c["id"]} for c in columns] if columns else [],
            style_table={'overflowX': 'auto'}
        )
    ]))

    return sections


if __name__ == '__main__':
    app.run(debug=True, port=9870)